In [ ]:
import numpy as np, pandas as pd, joblib, warnings, re, sys
from math import isfinite
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.calibration import CalibratedClassifierCV
from scipy.signal import savgol_filter, find_peaks, peak_widths
from scipy.optimize import curve_fit

# Config
CSV_PATH  = "../../data/shark_dataset.csv"
SPECIES_COL = "Species"

# Fitting / features
K_RANGE = (1, 6) # try 1..6 Gaussians, pick by BIC
DECIMATE_STEP = 6 # downsample factor for speed (lower => finer, slower)

# Model
CALIBRATE = True # probability calibration for nicer %'s
RANDOM_STATE = 8

# Outputs
MODEL_OUT = "gauss_species_model.joblib"
FEATURES_OUT = "gaussian_peak_features_all.csv"

In [ ]:
# Load data
df = pd.read_csv(CSV_PATH)
if SPECIES_COL not in df.columns:
    raise ValueError(f"Could not find '{SPECIES_COL}' in columns: {df.columns[:10]}...")
temp_cols = sorted([c for c in df.columns if c != SPECIES_COL], key=lambda c: float(c))
X_axis = np.array([float(c) for c in temp_cols], dtype=float)

print("Data shape:", df.shape)
print("Species counts:\n", df[SPECIES_COL].value_counts().sort_values(ascending=False).to_string())

In [ ]:
# Gaussians, preprocess, fitting
def gaussian(x, amp, mu, sigma):
    return amp * np.exp(-0.5 * ((x - mu) / (sigma + 1e-12)) ** 2)

def gaussian_sum(x, *p):
    y = np.zeros_like(x, dtype=float)
    for i in range(0, len(p), 3):
        amp, mu, sigma = p[i:i+3]
        y += gaussian(x, float(amp), float(mu), abs(float(sigma)))
    return y

def preprocess_curve(x, y):
    """Smooth + quadratic baseline removal + normalization."""
    y = np.asarray(y, float)
    dx = x[1] - x[0]
    win = max(7, int(round(1.5 / dx)) | 1)      # ~1.5°C window, odd
    if win >= len(y):                            # safety
        win = max(7, (len(y)//2)*2 - 1)
    y_s = savgol_filter(y, window_length=max(7, win), polyorder=3, mode="interp")

    q = np.quantile(y_s, 0.3)
    mask = y_s <= q
    if mask.sum() >= 10:
        coeffs = np.polyfit(x[mask], y_s[mask], deg=2)
        baseline = np.polyval(coeffs, x)
        y_b = y_s - baseline
    else:
        y_b = y_s - np.min(y_s)

    scale = np.quantile(y_b, 0.99)
    if scale > 0:
        y_b = y_b / scale
    y_b = np.maximum(y_b, 0.0)
    return y_b

def decimate(x, y, step=8):
    return x[::step], y[::step]

def seed_peaks(x, y, k):
    """Robust peak seeds: use prominence from spread, clamp widths."""
    spread = max(np.quantile(y, 0.90) - np.quantile(y, 0.10), 1e-6)
    prom = spread * 0.15
    peaks, props = find_peaks(y, prominence=prom, distance=max(1, len(x)//150))
    if len(peaks) == 0:
        peaks = np.argsort(y)[::-1][:k]
        prominences = y[peaks] - np.min(y)
    else:
        prominences = props["prominences"]

    if len(peaks) > 0:
        w_idx = peak_widths(y, peaks, rel_height=0.5)[0]
        w_c = w_idx * (x[1] - x[0])  # °C
    else:
        w_c = np.array([ (x[-1]-x[0])/(3*k) ]*k)

    min_w = max(0.2, 4*(x[1]-x[0]))
    max_w = (x[-1]-x[0]) / 3.0
    if np.ndim(w_c) == 0: w_c = np.array([w_c])
    w_c = np.clip(w_c, min_w, max_w)

    order = np.argsort(prominences)[::-1] if len(peaks) else np.arange(len(peaks))
    peaks = peaks[order][:k]
    w_c = w_c[order][:k]

    sort_lr = np.argsort(peaks)
    return peaks[sort_lr], w_c[sort_lr]

def fit_k(x, y, k):
    peaks, w_c = seed_peaks(x, y, k)
    p0, lo, hi = [], [], []
    y_max = max(np.max(y), 1e-6)
    for j, pk in enumerate(peaks):
        mu0 = float(x[pk])
        amp0 = float(max(y[pk], 1e-6))
        sigma0 = float(max(w_c[j] / (2*np.sqrt(2*np.log(2))), (x[1]-x[0])*2))
        p0 += [amp0, mu0, sigma0]
        lo += [0.0,   mu0 - 3*sigma0, (x[1]-x[0])*1e-3]
        hi += [y_max*5 + 1e-6, mu0 + 3*sigma0, (x[-1]-x[0])]
    popt, _ = curve_fit(gaussian_sum, x, y, p0=p0, bounds=(lo, hi), maxfev=15000)
    return popt

def BIC(n_params, rss, n):
    return np.log(n)*n_params + n*np.log(rss/n + 1e-12)

def fit_best_K(x, y, K=(1,5)):
    best = None
    for k in range(K[0], K[1]+1):
        try:
            popt = fit_k(x, y, k)
            yhat = gaussian_sum(x, *popt)
            rss = float(np.sum((y - yhat)**2))
            bic = float(BIC(3*k, rss, len(x)))
            if best is None or bic < best["bic"]:
                best = {"k": k, "popt": popt, "bic": bic}
        except Exception:
            pass
    return best

def peaks_to_features(popt, k_keep=2):
    peaks = [{"amp": float(popt[i]), "mu": float(popt[i+1]), "sigma": abs(float(popt[i+2]))}
             for i in range(0, len(popt), 3)]
    peaks.sort(key=lambda d: d["amp"], reverse=True)
    feats = {}
    for i in range(k_keep):
        if i < len(peaks):
            feats[f"peak{i+1}_mu"] = peaks[i]["mu"]
            feats[f"peak{i+1}_amp"] = peaks[i]["amp"]
            feats[f"peak{i+1}_sigma"] = peaks[i]["sigma"]
        else:
            feats[f"peak{i+1}_mu"] = feats[f"peak{i+1}_amp"] = feats[f"peak{i+1}_sigma"] = 0.0
    feats["delta_mu_12"] = feats["peak1_mu"] - feats["peak2_mu"]
    feats["amp_ratio_12"] = (feats["peak1_amp"]+1e-9)/(feats["peak2_amp"]+1e-9) if feats["peak2_amp"]>0 else 1e9
    return feats

def extra_features_from_fit(x, y, popt):
    # area per peak ~ amp * sigma * sqrt(2π)
    areas = []
    for i in range(0, len(popt), 3):
        amp, mu, sigma = float(popt[i]), float(popt[i+1]), abs(float(popt[i+2]))
        areas.append(amp * sigma * np.sqrt(2*np.pi))
    total_area = float(np.sum(areas)) if areas else 0.0

    # asymmetry around main peak from the preprocessed y
    main_mu = float(popt[1]) if len(popt)>=2 else x[np.argmax(y)]
    left  = y[(x >= main_mu-0.5) & (x <  main_mu)]
    right = y[(x >  main_mu)     & (x <= main_mu+0.5)]
    asym = (right.mean() - left.mean()) if (len(left)>3 and len(right)>3) else 0.0
    return {"total_area": total_area, "asym_0p5C": asym}

def extract_features_for_row(row_vals):
    """row_vals is a 1D array of curve values ordered by temp_cols."""
    y0 = preprocess_curve(X_axis, np.asarray(row_vals, float))
    x, y = decimate(X_axis, y0, step=DECIMATE_STEP)
    best = fit_best_K(x, y, K=K_RANGE)
    if best is None:
        return {
            "peak1_mu":0,"peak1_amp":0,"peak1_sigma":0,
            "peak2_mu":0,"peak2_amp":0,"peak2_sigma":0,
            "delta_mu_12":0,"amp_ratio_12":0,"best_K":0,
            "total_area":0.0,"asym_0p5C":0.0,"total_amp":0.0
        }
    feats = peaks_to_features(best["popt"], k_keep=2)
    extras = extra_features_from_fit(x, y, best["popt"])
    feats.update(extras)
    feats["best_K"] = best["k"]
    feats["total_amp"] = feats["peak1_amp"] + feats["peak2_amp"]
    return feats


In [ ]:
# Feature Extraction
rows = []
for i in range(len(df)):
    feats = extract_features_for_row(df.loc[i, temp_cols].values)
    feats[SPECIES_COL] = df.loc[i, SPECIES_COL]
    rows.append(feats)

feat_df = pd.DataFrame(rows).fillna(0.0)
feat_df.to_csv(FEATURES_OUT, index=False)
print("Saved features ->", FEATURES_OUT)
print("Feature columns:", [c for c in feat_df.columns if c != SPECIES_COL][:20], "...")


Saved features -> /content/gaussian_peak_features_all.csv
Feature columns: ['peak1_mu', 'peak1_amp', 'peak1_sigma', 'peak2_mu', 'peak2_amp', 'peak2_sigma', 'delta_mu_12', 'amp_ratio_12', 'total_area', 'asym_0p5C', 'best_K', 'total_amp'] ...


In [ ]:
# Train / Eval / Save
X = feat_df.drop(columns=[SPECIES_COL]).to_numpy(float)
y = df[SPECIES_COL].astype(str).to_numpy()

min_class = df[SPECIES_COL].value_counts().min()
n_splits = max(2, min(4, int(min_class)))  # avoid > min_class
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

base_rf = RandomForestClassifier(
    n_estimators=800,
    random_state=RANDOM_STATE,
    class_weight="balanced_subsample",
    max_depth=None,
    min_samples_leaf=1
)

clf = CalibratedClassifierCV(base_rf, cv=min(3, n_splits), method="isotonic") if CALIBRATE else base_rf

scores = cross_val_score(clf, X, y, cv=cv, scoring="accuracy")
print(f"CV (n_splits={n_splits}) accuracies:", np.round(scores, 3), "mean:", np.round(scores.mean(), 3))

# Hold-out snapshot
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
clf.fit(Xtr, ytr)
pred = clf.predict(Xte)
print("\n=== Holdout report ===")
print(classification_report(yte, pred, digits=3, zero_division=0))
cm = confusion_matrix(yte, pred, labels=np.unique(y))
print("Confusion matrix shape:", cm.shape)

# Save
bundle = {
    "model": clf,
    "feature_names": feat_df.drop(columns=[SPECIES_COL]).columns.tolist(),
    "classes_": np.unique(y)
}
joblib.dump(bundle, MODEL_OUT)
print("Saved model ->", MODEL_OUT)


CV (n_splits=4) accuracies: [0.896 0.877 0.883 0.901] mean: 0.889

=== Holdout report ===
                            precision    recall  f1-score   support

      Arabian smooth-hound      1.000     1.000     1.000         1
  Atlantic Sharpnose shark      1.000     1.000     1.000         1
      Blackchin guitarfish      1.000     1.000     1.000         2
           Blacknose shark      0.000     0.000     0.000         1
 Blackspotted smooth-hound      0.000     0.000     0.000         1
       Blacktip reef shark      0.000     0.000     0.000         1
            Blacktip shark      0.750     1.000     0.857         6
                Blue shark      1.000     0.714     0.833         7
          Bonnethead shark      1.000     1.000     1.000         1
       Bowmouth guitarfish      1.000     1.000     1.000         1
  Brownbanded bamboo shark      1.000     1.000     1.000         2
                Bull shark      1.000     0.833     0.909         6
      Caribbean reef shar

In [ ]:
# =======================
# FULL METRICS & FIGURES SUITE
# (run after training/eval; expects df, X, y, Xtr, Xte, ytr, yte, pred, bundle, temp_cols, X_axis,
#  and the Gaussian helpers defined earlier)
# =======================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)
from sklearn.calibration import calibration_curve

# -------- pick the trained model ----------
model = bundle["model"]  # this is your classifier (RF or calibrated RF or XGB)
feature_names = bundle.get("feature_names", list(range(X.shape[1])))

# -------- probabilities on holdout ----------
proba = model.predict_proba(Xte)
classes = getattr(model, "classes_", None)
# If using XGB with label encoder in bundle:
if classes is None and "label_encoder_classes_" in bundle:
    classes = bundle["label_encoder_classes_"]

# -------- overall scalar metrics ----------
acc = accuracy_score(yte, pred)
macro_f1 = f1_score(yte, pred, average="macro", zero_division=0)
weighted_f1 = f1_score(yte, pred, average="weighted", zero_division=0)

# Top-3 accuracy
topk = 3
topk_idx = np.argsort(proba, axis=1)[:, -topk:]
# Map true labels into index space if classes are strings
if isinstance(yte[0], str):
    # Ensure classes is sorted like classifier expects
    idx_map = {c:i for i,c in enumerate(classes)}
    y_idx = np.array([idx_map[v] for v in yte], dtype=int)
else:
    # yte already integer-coded aligned with classes
    # but if classes are strings and yte ints, also fine
    y_idx = yte
top3_acc = np.mean([y_idx[i] in topk_idx[i] for i in range(len(yte))])

# Abstain@0.60
threshold = 0.60
maxp = proba.max(axis=1)
keep = maxp >= threshold
coverage = keep.mean()
accepted_acc = (pred[keep] == np.array(yte)[keep]).mean() if keep.any() else np.nan

# Print a compact metrics table
metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Macro F1", "Weighted F1", "Top-3 Accuracy", f"Coverage @ {threshold:.2f}", f"Accepted Acc @ {threshold:.2f}"],
    "Value":  [acc,        macro_f1,   weighted_f1,    top3_acc,           coverage,                    accepted_acc]
})
print("\n=== OVERALL METRICS (Hold-out) ===")
print(metrics_df.to_string(index=False))

# If you still have the CV scores in 'scores' or logged earlier, you can reuse them.
# Otherwise, set cv_scores to an empty list and the plot will be skipped gracefully.
cv_scores = globals().get("scores", None)  # from your earlier cross_val_score call (RF path)
if cv_scores is not None and hasattr(cv_scores, "__len__"):
    cv_scores = np.array(cv_scores)
    print("\nCV fold accuracies:", np.round(cv_scores, 3), "mean:", np.round(cv_scores.mean(), 3))
else:
    print("\n(No CV scores array named 'scores' found; skipping CV plot.)")

# -------- 1) CV fold accuracies plot --------
if cv_scores is not None and len(cv_scores) > 0:
    plt.figure(figsize=(5,3))
    plt.bar(np.arange(1, len(cv_scores)+1), cv_scores)
    plt.axhline(cv_scores.mean(), linestyle="--")
    plt.title("Cross-Validation Accuracies")
    plt.xlabel("Fold")
    plt.ylabel("Accuracy")
    plt.tight_layout()
    plt.show()

# -------- 2) Confusion matrix (normalized) --------
labels_for_cm = np.array(classes)
cm = confusion_matrix(yte, pred, labels=labels_for_cm)
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-12)

fig, ax = plt.subplots(figsize=(8,7))
im = ax.imshow(cm_norm, aspect="auto")
ax.set_title("Confusion Matrix (Row-Normalized)")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
# show fewer ticks if many classes
step = max(1, len(labels_for_cm)//20)
ax.set_xticks(np.arange(0, len(labels_for_cm), step))
ax.set_yticks(np.arange(0, len(labels_for_cm), step))
ax.set_xticklabels(labels_for_cm[::step], rotation=90)
ax.set_yticklabels(labels_for_cm[::step])
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

# -------- 3) Per-class precision/recall/F1 bar charts --------
prec, rec, f1, support = precision_recall_fscore_support(yte, pred, labels=labels_for_cm, zero_division=0)
per_class = pd.DataFrame({
    "species": labels_for_cm,
    "precision": prec,
    "recall": rec,
    "f1": f1,
    "support": support
}).sort_values("support", ascending=False)

# Plot (maybe only top N by support for readability)
N = min(25, len(per_class))  # show top 25 by support
subset = per_class.head(N)

fig, ax = plt.subplots(3,1, figsize=(10,10), sharex=True)
ax[0].bar(subset["species"], subset["precision"])
ax[0].set_title("Per-Class Precision (top by support)")
ax[1].bar(subset["species"], subset["recall"])
ax[1].set_title("Per-Class Recall (top by support)")
ax[2].bar(subset["species"], subset["f1"])
ax[2].set_title("Per-Class F1 (top by support)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# -------- 4) Reliability diagram + confidence histogram --------
# Build true correctness and predicted confidence
correct = (pred == yte).astype(int)
conf = maxp

# Calibration curve: use bins on predicted max probability
prob_true, prob_pred = calibration_curve(correct, conf, n_bins=10, strategy="uniform")

fig, ax = plt.subplots(1,2, figsize=(10,4))
# Reliability
ax[0].plot([0,1], [0,1], linestyle="--")
ax[0].plot(prob_pred, prob_true, marker="o")
ax[0].set_title("Reliability Diagram (Max Prob as Confidence)")
ax[0].set_xlabel("Predicted probability")
ax[0].set_ylabel("Empirical accuracy")

# Histogram of confidences
ax[1].hist(conf, bins=20)
ax[1].set_title("Histogram of Max Predicted Probabilities")
ax[1].set_xlabel("Max probability")
ax[1].set_ylabel("Count")
plt.tight_layout()
plt.show()

# -------- 5) Feature importance (works for RF; best-effort for calibrated RF) --------
def extract_rf_feature_importances(m):
    """Try to pull RF feature importances even if wrapped in CalibratedClassifierCV."""
    # Direct RF
    if hasattr(m, "feature_importances_"):
        return m.feature_importances_
    # CalibratedClassifierCV: try first calibrator's estimator if it's an RF
    if hasattr(m, "calibrated_classifiers_") and len(m.calibrated_classifiers_)>0:
        est = getattr(m.calibrated_classifiers_[0], "estimator", None)
        if est is not None and hasattr(est, "feature_importances_"):
            return est.feature_importances_
    return None

fi = extract_rf_feature_importances(model)
if fi is not None:
    order = np.argsort(fi)[::-1]
    topn = min(20, len(fi))
    plt.figure(figsize=(8,5))
    plt.barh(np.array(feature_names)[order[:topn]][::-1], np.array(fi)[order[:topn]][::-1])
    plt.title("Feature Importances (top 20)")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
else:
    print("\n(Feature importances not available for this model type.)")

# -------- 6) Example curves with Gaussian fits (3 random samples) --------
def plot_fit_example(idx):
    # grab raw curve
    y_raw = df.loc[idx, temp_cols].values.astype(float)
    y_pre = preprocess_curve(X_axis, y_raw)
    x_d, y_d = decimate(X_axis, y_pre, step=DECIMATE_STEP)
    best = fit_best_K(x_d, y_d, K=K_RANGE)
    plt.figure(figsize=(7,4))
    plt.plot(X_axis, y_pre, label="preprocessed")
    if best is not None:
        popt = best["popt"]
        yhat = gaussian_sum(x_d, *popt)
        # plot fitted sum (on decimated x) and individual components
        plt.plot(x_d, yhat, linewidth=2, label=f"fit sum (K={best['k']})")
        for i in range(0, len(popt), 3):
            amp, mu, sigma = popt[i], popt[i+1], abs(popt[i+2])
            plt.plot(x_d, gaussian(x_d, amp, mu, sigma), linestyle=":")
    plt.title(f"Row {idx} | True: {df.loc[idx, SPECIES_COL]}")
    plt.xlabel("Temperature (°C)")
    plt.ylabel("Normalized signal")
    plt.legend()
    plt.tight_layout()
    plt.show()

print("\n=== Example curve fits ===")
for idx in np.random.choice(df.index, size=3, replace=False):
    plot_fit_example(int(idx))


NameError: name 'bundle' is not defined

In [ ]:
# Inference helpers
def predict_with_probs(curve_1d, top_n=5):
    """Returns (best_species, best_prob, [(species, prob), ...sorted])"""
    feats = extract_features_for_row(curve_1d)
    X1 = pd.DataFrame([feats])[bundle["feature_names"]].to_numpy(float)
    probs = bundle["model"].predict_proba(X1)[0]
    classes = bundle["model"].classes_
    ranking = sorted(zip(classes, probs), key=lambda t: t[1], reverse=True)
    best_label, best_p = ranking[0]
    top = [(lbl, float(p)) for lbl, p in ranking[:top_n]]
    return best_label, float(best_p), top


In [1]:
# Random tests
n_samples = 5
for idx in np.random.choice(df.index, size=min(n_samples, len(df)), replace=False):
    curve = df.loc[idx, temp_cols].values
    true_species = df.loc[idx, SPECIES_COL]
    best, p, top = predict_with_probs(curve, top_n=5)
    print(f"\nRow {idx} | True: {true_species} | Pred: {best} ({p*100:.1f}%)")
    for lbl, pr in top:
        print(f"  {lbl:30s} {pr*100:5.3f}%")


NameError: name 'np' is not defined

In [ ]:
# Metrics
proba = bundle["model"].predict_proba(Xte)
classes = bundle["model"].classes_
true_idx = np.searchsorted(classes, yte)
top3_idx = np.argsort(proba, axis=1)[:, -3:]
top3_acc = np.mean([true_idx[i] in top3_idx[i] for i in range(len(yte))])
print("Top-3 accuracy (holdout):", round(top3_acc, 3))

threshold = 0.60
maxp = proba.max(axis=1)
keep = maxp >= threshold
cov = keep.mean()
acc = (pred[keep] == yte[keep]).mean() if keep.any() else float("nan")
print(f"Abstain@{threshold:.2f}: coverage={cov:.2%}, accepted-accuracy={acc:.3f}")


Top-3 accuracy (holdout): 0.939
Abstain@0.60: coverage=86.26%, accepted-accuracy=0.947


In [ ]:
# # Export
# from google.colab import files
# files.download(MODEL_OUT)
# files.download(FEATURES_OUT)